# CDT-III: Data Preparation (DSB Normalization + RNA+ADT Integration)

## Steps
1. **DSB正規化**: `stingseq_v2_with_adt.h5` → isotype controlでバックグラウンド除去
2. **RNA+ADT統合**: `morris_celllevel_effects_2361.h5` + DSB正規化ADT → 統合データセット
3. **検証**: データ範囲、NTCベースライン、キーマーカー確認

## Input Files (Google Drive)
- `stingseq_v2_with_adt.h5` — Raw ADT counts + RNA (605MB)
- `morris_celllevel_effects_2361.h5` — Cell-level RNA log2FC

## Output
- `morris_celllevel_effects_with_adt.h5` — CDT-III訓練用統合データ

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import h5py
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

DRIVE_BASE = Path("/content/drive/MyDrive/cdt_data")

# Input files
ADT_H5_PATH = DRIVE_BASE / "stingseq_v2_with_adt.h5"
RNA_CELLLEVEL_PATH = DRIVE_BASE / "morris_celllevel_effects_2361.h5"

# Output
OUTPUT_PATH = DRIVE_BASE / "morris_celllevel_effects_with_adt.h5"

print("File check:")
for name, path in [("ADT H5", ADT_H5_PATH), ("RNA cell-level", RNA_CELLLEVEL_PATH)]:
    if path.exists():
        size_mb = path.stat().st_size / (1024**2)
        print(f"  OK  {name}: {path.name} ({size_mb:.0f} MB)")
    else:
        print(f"  NG  {name}: {path.name} NOT FOUND")

## Step 1: Load & Inspect ADT Data

In [ ]:
# ============================================================
# Load ADT data
# ============================================================
print("Loading ADT data...")
with h5py.File(ADT_H5_PATH, 'r') as f:
    print("  Keys:", list(f.keys()))
    print("  Attrs:", dict(f.attrs))
    
    adt_counts = f['adt_counts'][:]    # [n_cells, n_adt]
    adt_names = [n.decode() if isinstance(n, bytes) else n
                 for n in f['adt_names'][:]]
    cell_barcodes_adt = [b.decode() if isinstance(b, bytes) else b
                         for b in f['cell_barcodes'][:]]
    
    # Load guide info for NTC identification
    if 'guide_counts' in f:
        guide_counts = f['guide_counts'][:]
        guide_names = [g.decode() if isinstance(g, bytes) else g
                       for g in f['guide_names'][:]]
    else:
        guide_counts = None
        guide_names = []

n_cells_adt, n_adt = adt_counts.shape
print(f"\nADT data:")
print(f"  Cells: {n_cells_adt:,}")
print(f"  ADT features: {n_adt}")
print(f"  ADT names: {adt_names[:5]} ... {adt_names[-5:]}")
print(f"  Count range: [{adt_counts.min():.0f}, {adt_counts.max():.0f}]")
print(f"  Count mean: {adt_counts.mean():.1f}")

In [ ]:
# ============================================================
# Identify Isotype Controls & Proteins
# ============================================================
isotype_indices = []
isotype_names = []
protein_indices = []
protein_names = []

for i, name in enumerate(adt_names):
    if name.startswith('ADT-CTL'):
        isotype_indices.append(i)
        isotype_names.append(name)
    else:
        protein_indices.append(i)
        protein_names.append(name)

print(f"Isotype controls ({len(isotype_indices)}): {isotype_names}")
print(f"Proteins: {len(protein_indices)}")

# Isotype control stats
print(f"\nIsotype control raw counts:")
for idx, name in zip(isotype_indices, isotype_names):
    vals = adt_counts[:, idx]
    print(f"  {name}: mean={vals.mean():.1f}, median={np.median(vals):.1f}, "
          f"std={vals.std():.1f}, max={vals.max():.0f}")

# A few protein stats
print(f"\nSelected protein raw counts:")
for adt_id, label in [('ADT-0394', 'CD71/TFRC'), ('ADT-0033', 'CD52'),
                       ('ADT-0034', 'CD3'), ('ADT-0046', 'CD8')]:
    if adt_id in adt_names:
        idx = adt_names.index(adt_id)
        vals = adt_counts[:, idx]
        print(f"  {adt_id} ({label}): mean={vals.mean():.1f}, "
              f"median={np.median(vals):.1f}, max={vals.max():.0f}")

## Step 2: DSB Normalization

In [ ]:
# ============================================================
# DSB Normalization
# ============================================================
# DSB(x) = (log(x + pseudocount) - mu_bg) / sigma_bg
# where mu_bg, sigma_bg come from isotype controls per cell
# Reference: Mulè et al., Nature Immunology 2022

PSEUDOCOUNT = 10.0  # DSB paper recommendation

print("Running DSB normalization...")

# Step 1: Log-transform
log_counts = np.log(adt_counts + PSEUDOCOUNT)

# Step 2: Per-cell background from isotype controls
isotype_vals = log_counts[:, isotype_indices]  # [n_cells, 4]
mu_bg = np.mean(isotype_vals, axis=1, keepdims=True)    # [n_cells, 1]
sigma_bg = np.std(isotype_vals, axis=1, keepdims=True)  # [n_cells, 1]
sigma_bg = np.maximum(sigma_bg, 0.01)  # prevent division by zero

# Step 3: Normalize
adt_dsb = (log_counts - mu_bg) / sigma_bg

# Extract protein-only (exclude isotype controls)
adt_dsb_proteins = adt_dsb[:, protein_indices].astype(np.float32)

print(f"\nDSB normalization results:")
print(f"  Shape: {adt_dsb_proteins.shape}")
print(f"  Background mu: mean={mu_bg.mean():.3f}, std={mu_bg.std():.3f}")
print(f"  Background sigma: mean={sigma_bg.mean():.3f}")
print(f"  Isotype post-DSB mean: {adt_dsb[:, isotype_indices].mean():.4f} (should be ~0)")
print(f"  Proteins post-DSB: mean={adt_dsb_proteins.mean():.3f}, std={adt_dsb_proteins.std():.3f}")
print(f"  Proteins range: [{adt_dsb_proteins.min():.2f}, {adt_dsb_proteins.max():.2f}]")

In [ ]:
# ============================================================
# Validation: DSB値の分布確認
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Isotype controls
for idx, name in zip(isotype_indices, isotype_names):
    axes[0, 0].hist(adt_dsb[:, idx], bins=50, alpha=0.5, label=name)
axes[0, 0].set_title('Isotype Controls (post-DSB)')
axes[0, 0].set_xlabel('DSB value')
axes[0, 0].axvline(x=0, color='red', linestyle='--')
axes[0, 0].legend(fontsize=8)

# Key proteins
markers = [('ADT-0394', 'CD71/TFRC'), ('ADT-0033', 'CD52'),
           ('ADT-0034', 'CD3'), ('ADT-0046', 'CD8')]
for adt_id, label in markers:
    if adt_id in adt_names:
        idx = adt_names.index(adt_id)
        axes[0, 1].hist(adt_dsb[:, idx], bins=50, alpha=0.5, label=label)
axes[0, 1].set_title('Key Proteins (post-DSB)')
axes[0, 1].set_xlabel('DSB value')
axes[0, 1].axvline(x=0, color='red', linestyle='--')
axes[0, 1].legend(fontsize=8)

# Per-protein mean DSB
protein_means = adt_dsb_proteins.mean(axis=0)
axes[1, 0].bar(range(len(protein_means)), np.sort(protein_means)[::-1], width=1.0)
axes[1, 0].set_xlabel('Protein (sorted by mean DSB)')
axes[1, 0].set_ylabel('Mean DSB')
axes[1, 0].set_title(f'Per-protein Mean DSB ({len(protein_means)} proteins)')
axes[1, 0].axhline(y=0, color='red', linestyle='--')

# Overall distribution
flat = adt_dsb_proteins.flatten()
axes[1, 1].hist(flat[::100], bins=100, alpha=0.7, color='steelblue')  # subsample for speed
axes[1, 1].set_title('Overall DSB Distribution (subsampled)')
axes[1, 1].set_xlabel('DSB value')
axes[1, 1].axvline(x=0, color='red', linestyle='--')

plt.suptitle('DSB Normalization Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print summary stats per key protein
print("\nKey protein DSB values:")
for adt_id, label in markers:
    if adt_id in adt_names:
        idx = adt_names.index(adt_id)
        vals = adt_dsb[:, idx]
        print(f"  {label}: mean={vals.mean():.3f}, std={vals.std():.3f}, "
              f"median={np.median(vals):.3f}")

## Step 3: Identify NTC Cells

In [ ]:
# ============================================================
# Identify NTC (non-targeting control) cells
# ============================================================
print("Identifying NTC cells...")

if guide_counts is not None and len(guide_names) > 0:
    ntc_guide_indices = [i for i, g in enumerate(guide_names)
                         if 'NTC' in g.upper() or 'NON-TARGETING' in g.upper()
                         or 'NON_TARGETING' in g.upper()]
    print(f"  NTC guide indices: {len(ntc_guide_indices)}")
    if ntc_guide_indices:
        print(f"  Examples: {[guide_names[i] for i in ntc_guide_indices[:5]]}")
    
    # Cells with NTC guides
    ntc_mask = np.zeros(n_cells_adt, dtype=bool)
    for gi in ntc_guide_indices:
        ntc_mask |= guide_counts[:, gi] > 0
    
    n_ntc = ntc_mask.sum()
    print(f"  NTC cells: {n_ntc:,} / {n_cells_adt:,} ({100*n_ntc/n_cells_adt:.1f}%)")
else:
    print("  WARNING: No guide info found. Using global mean as NTC proxy.")
    ntc_mask = np.ones(n_cells_adt, dtype=bool)  # fallback: all cells

# NTC mean protein DSB
ntc_mean_protein = adt_dsb_proteins[ntc_mask].mean(axis=0)  # [189]
ntc_std_protein = adt_dsb_proteins[ntc_mask].std(axis=0)

print(f"\nNTC protein DSB stats:")
print(f"  Shape: {ntc_mean_protein.shape}")
print(f"  Mean of means: {ntc_mean_protein.mean():.3f}")
print(f"  Range: [{ntc_mean_protein.min():.3f}, {ntc_mean_protein.max():.3f}]")

## Step 4: Load RNA Cell-Level Effects & Match Cells

In [ ]:
# ============================================================
# Load RNA cell-level effects
# ============================================================
print("Loading RNA cell-level effects...")

with h5py.File(RNA_CELLLEVEL_PATH, 'r') as f:
    print("  Keys:", list(f.keys()))
    print("  Attrs:", dict(f.attrs))
    
    rna_log2fc = f['log2fc'][:]              # [n_rna_cells, 2361]
    cell_expr = f['cell_expr'][:]            # [n_rna_cells, 2361]
    target_gene_idx = f['target_gene_idx'][:]
    cdt_genes = [g.decode() if isinstance(g, bytes) else g
                 for g in f['gene_names_cdt'][:]]
    target_gene_names = [g.decode() if isinstance(g, bytes) else g
                         for g in f['target_gene_names'][:]]
    ntc_mean_expr = f['ntc_mean_expr'][:]
    train_genes = [g.decode() if isinstance(g, bytes) else g
                   for g in f['train_genes'][:]]
    val_genes = [g.decode() if isinstance(g, bytes) else g
                 for g in f['val_genes'][:]]
    n_cells_per_gene = f['n_cells_per_gene'][:]
    
    # Cell indices mapping back to original file
    if 'original_cell_idx' in f:
        original_cell_idx = f['original_cell_idx'][:]
    else:
        original_cell_idx = None
    
    if 'cell_barcodes' in f:
        rna_barcodes = [bc.decode() if isinstance(bc, bytes) else bc
                        for bc in f['cell_barcodes'][:]]
    else:
        rna_barcodes = None
    
    n_rna_cells = f.attrs['n_cells']
    n_ntc_rna = f.attrs['n_ntc_cells']

n_genes = len(cdt_genes)
print(f"\nRNA data:")
print(f"  Cells: {n_rna_cells:,}")
print(f"  Genes: {n_genes}")
print(f"  Train genes: {len(train_genes)}")
print(f"  Val genes: {val_genes}")
print(f"  original_cell_idx: {'available' if original_cell_idx is not None else 'not available'}")
print(f"  cell_barcodes: {'available' if rna_barcodes is not None else 'not available'}")

In [ ]:
# ============================================================
# Match RNA cells to ADT cells
# ============================================================
print("Matching RNA cells → ADT cells...")

rna_to_adt_map = {}  # rna_cell_idx → adt_cell_idx

if rna_barcodes is not None:
    # Method 1: Barcode matching
    print("  Using barcode matching...")
    adt_bc_to_idx = {bc: i for i, bc in enumerate(cell_barcodes_adt)}
    
    for rna_idx, bc in enumerate(rna_barcodes):
        # Try exact match and common format variations
        candidates = [bc]
        if '_' in bc:
            candidates.append(bc.split('_')[0])
        if not bc.endswith('-1'):
            candidates.append(f"{bc}-1")
        
        for c in candidates:
            if c in adt_bc_to_idx:
                rna_to_adt_map[rna_idx] = adt_bc_to_idx[c]
                break
    
    print(f"  Barcode matches: {len(rna_to_adt_map)}/{n_rna_cells}")

elif original_cell_idx is not None:
    # Method 2: original_cell_idx mapping
    print("  Using original_cell_idx mapping...")
    for rna_idx in range(n_rna_cells):
        adt_idx = int(original_cell_idx[rna_idx])
        if adt_idx < n_cells_adt:
            rna_to_adt_map[rna_idx] = adt_idx
    print(f"  Index matches: {len(rna_to_adt_map)}/{n_rna_cells}")

else:
    print("  WARNING: No matching method available!")
    print("  Trying direct index mapping (same ordering assumed)...")
    for i in range(min(n_rna_cells, n_cells_adt)):
        rna_to_adt_map[i] = i

n_matched = len(rna_to_adt_map)
print(f"\nMatched: {n_matched:,} / {n_rna_cells:,} ({100*n_matched/n_rna_cells:.1f}%)")

## Step 5: Compute Protein Effect (DSB Difference)

In [ ]:
# ============================================================
# Build protein data matrix for RNA cells
# ============================================================
n_proteins = len(protein_indices)

protein_dsb_matched = np.zeros((n_rna_cells, n_proteins), dtype=np.float32)
protein_matched_mask = np.zeros(n_rna_cells, dtype=bool)

for rna_idx, adt_idx in rna_to_adt_map.items():
    protein_dsb_matched[rna_idx] = adt_dsb_proteins[adt_idx]
    protein_matched_mask[rna_idx] = True

n_with_protein = protein_matched_mask.sum()
print(f"Cells with protein data: {n_with_protein:,} / {n_rna_cells:,}")

# ============================================================
# Compute protein effect = cell_DSB - NTC_mean_DSB
# ============================================================
print("\nComputing protein effect (DSB difference)...")
protein_effect = protein_dsb_matched - ntc_mean_protein[np.newaxis, :]
protein_effect[~protein_matched_mask] = 0.0  # unmatched cells → 0

matched_effects = protein_effect[protein_matched_mask]
print(f"\nProtein effect stats (matched cells only):")
print(f"  Shape: {matched_effects.shape}")
print(f"  Mean: {matched_effects.mean():.4f}")
print(f"  Std: {matched_effects.std():.4f}")
print(f"  Range: [{matched_effects.min():.3f}, {matched_effects.max():.3f}]")
print(f"  |effect| > 0.5: {(np.abs(matched_effects) > 0.5).mean()*100:.1f}%")
print(f"  |effect| > 1.0: {(np.abs(matched_effects) > 1.0).mean()*100:.1f}%")

# Compare with RNA log2FC range
rna_matched = rna_log2fc[protein_matched_mask]
print(f"\nRNA log2FC stats (for comparison):")
print(f"  Mean: {rna_matched.mean():.4f}")
print(f"  Std: {rna_matched.std():.4f}")
print(f"  Range: [{rna_matched.min():.3f}, {rna_matched.max():.3f}]")

In [ ]:
# ============================================================
# Validation: Key markers
# ============================================================
print("Key marker protein effects:")

prot_name_to_idx = {name: i for i, name in enumerate(protein_names)}

key_markers = [
    ('ADT-0394', 'CD71/TFRC', 'High in K562 (iron metabolism)'),
    ('ADT-0033', 'CD52', 'CRISPRi target, variable'),
    ('ADT-0034', 'CD3/CD3E', 'T-cell marker, should be ~0 in K562'),
    ('ADT-0046', 'CD8/CD8A', 'T-cell marker, should be ~0 in K562'),
    ('ADT-0057', 'B2M', 'HLA component, ubiquitous'),
]

for adt_id, label, note in key_markers:
    if adt_id in prot_name_to_idx:
        idx = prot_name_to_idx[adt_id]
        dsb_vals = protein_dsb_matched[protein_matched_mask, idx]
        eff_vals = protein_effect[protein_matched_mask, idx]
        ntc_val = ntc_mean_protein[idx]
        print(f"\n  {adt_id} ({label}): {note}")
        print(f"    NTC mean DSB: {ntc_val:.3f}")
        print(f"    All cells DSB: mean={dsb_vals.mean():.3f}, std={dsb_vals.std():.3f}")
        print(f"    Effect (DSB diff): mean={eff_vals.mean():.4f}, std={eff_vals.std():.4f}")

In [ ]:
# ============================================================
# Per-perturbation protein effect distribution
# ============================================================
print("Per-perturbation protein effect (val genes):")

gene_name_to_target_idx = {n: i for i, n in enumerate(target_gene_names)}

fig, axes = plt.subplots(1, len(val_genes), figsize=(4*len(val_genes), 4))
if len(val_genes) == 1:
    axes = [axes]

for ax, gene in zip(axes, val_genes):
    if gene not in gene_name_to_target_idx:
        continue
    g_idx = gene_name_to_target_idx[gene]
    
    # Get cells for this gene
    gene_cells = np.where(
        (target_gene_idx == g_idx) & protein_matched_mask)[0]
    
    if len(gene_cells) == 0:
        print(f"  {gene}: no matched cells")
        continue
    
    gene_effects = protein_effect[gene_cells]  # [n_cells, 189]
    mean_effect = gene_effects.mean(axis=0)     # [189]
    
    print(f"  {gene}: {len(gene_cells)} cells, "
          f"mean |effect|={np.abs(mean_effect).mean():.4f}, "
          f"max |effect|={np.abs(mean_effect).max():.4f}")
    
    # Histogram of mean effects across proteins
    ax.hist(mean_effect, bins=30, alpha=0.7, color='steelblue')
    ax.axvline(x=0, color='red', linestyle='--')
    ax.set_title(f'{gene} KD\n({len(gene_cells)} cells)')
    ax.set_xlabel('Mean protein effect')
    
    # Mark the gene's own protein if it has one
    for adt_id, label in [('ADT-0394', 'CD71'), ('ADT-0033', 'CD52')]:
        if adt_id in prot_name_to_idx and gene in label:
            own_idx = prot_name_to_idx[adt_id]
            ax.axvline(x=mean_effect[own_idx], color='green',
                      linestyle='-', linewidth=2, label=f'{label}')
            ax.legend(fontsize=8)

plt.suptitle('Protein Effect Distribution per Perturbation (DSB diff)', fontweight='bold')
plt.tight_layout()
plt.show()

## Step 6: Save Integrated Dataset

In [ ]:
# ============================================================
# Load ADT mapping for overlap analysis
# ============================================================
# Clone repo if needed for mapping file
import os
REPO_PATH = Path("/content/CDT-private")
MAPPING_PATH = REPO_PATH / "data/processed/morris/adt_mapping.csv"

if not MAPPING_PATH.exists():
    print("ADT mapping file not found locally.")
    print("Please clone repo or upload adt_mapping.csv")
    # Build minimal mapping from known proteins
    overlap_adt_ids = []
    overlap_genes = []
else:
    adt_mapping = pd.read_csv(MAPPING_PATH)
    adt_mapping = adt_mapping[~adt_mapping['morris_adt_id'].str.startswith('ADT-CTL')]
    
    cdt_gene_set = set(cdt_genes)
    overlapping = []
    for _, row in adt_mapping.iterrows():
        if row['morris_adt_id'] in protein_names and row['gene_symbol'] in cdt_gene_set:
            overlapping.append((row['morris_adt_id'], row['gene_symbol']))
    
    overlap_adt_ids = [o[0] for o in overlapping]
    overlap_genes = [o[1] for o in overlapping]
    print(f"RNA-Protein overlapping genes: {len(overlapping)}")
    for adt_id, gene in overlapping[:10]:
        print(f"  {adt_id} → {gene}")

In [ ]:
# ============================================================
# Save integrated dataset
# ============================================================
print(f"Saving integrated dataset to {OUTPUT_PATH}...")

with h5py.File(OUTPUT_PATH, 'w') as f:
    # RNA data
    f.create_dataset('log2fc', data=rna_log2fc, compression='gzip')
    f.create_dataset('cell_expr', data=cell_expr, compression='gzip')
    f.create_dataset('target_gene_idx', data=target_gene_idx)
    f.create_dataset('gene_names_cdt', data=np.array(cdt_genes, dtype='S'))
    f.create_dataset('target_gene_names', data=np.array(target_gene_names, dtype='S'))
    f.create_dataset('ntc_mean_expr', data=ntc_mean_expr)
    f.create_dataset('train_genes', data=np.array(train_genes, dtype='S'))
    f.create_dataset('val_genes', data=np.array(val_genes, dtype='S'))
    f.create_dataset('n_cells_per_gene', data=n_cells_per_gene)
    
    if rna_barcodes is not None:
        f.create_dataset('cell_barcodes', data=np.array(rna_barcodes, dtype='S'))
    if original_cell_idx is not None:
        f.create_dataset('original_cell_idx', data=original_cell_idx)
    
    # Protein data (NEW)
    f.create_dataset('protein_effect', data=protein_effect, compression='gzip')
    f.create_dataset('protein_dsb', data=protein_dsb_matched, compression='gzip')
    f.create_dataset('protein_names', data=np.array(protein_names, dtype='S'))
    f.create_dataset('ntc_mean_protein', data=ntc_mean_protein)
    f.create_dataset('protein_matched_mask', data=protein_matched_mask)
    
    # Overlap info
    if overlap_adt_ids:
        f.create_dataset('overlap_adt_ids', data=np.array(overlap_adt_ids, dtype='S'))
        f.create_dataset('overlap_gene_symbols', data=np.array(overlap_genes, dtype='S'))
    
    # Metadata
    f.attrs['n_cells'] = n_rna_cells
    f.attrs['n_genes'] = n_genes
    f.attrs['n_proteins'] = n_proteins
    f.attrs['n_cells_with_protein'] = int(n_with_protein)
    f.attrs['n_ntc_cells'] = int(n_ntc_rna)
    f.attrs['n_overlapping_genes'] = len(overlap_adt_ids)
    f.attrs['protein_normalization'] = 'DSB'
    f.attrs['protein_target'] = 'DSB_difference'

file_size = OUTPUT_PATH.stat().st_size / (1024**2)
print(f"\nSaved! Size: {file_size:.1f} MB")

# Verify
print("\nVerification:")
with h5py.File(OUTPUT_PATH, 'r') as f:
    print(f"  Keys: {list(f.keys())}")
    print(f"  Attrs: {dict(f.attrs)}")
    print(f"  log2fc: {f['log2fc'].shape}")
    print(f"  protein_effect: {f['protein_effect'].shape}")
    print(f"  protein_dsb: {f['protein_dsb'].shape}")

In [ ]:
# ============================================================
# Final Summary
# ============================================================
print("=" * 60)
print("CDT-III Data Preparation Complete!")
print("=" * 60)
print(f"")
print(f"  RNA:     {n_rna_cells:,} cells x {n_genes} genes")
print(f"  Protein: {n_with_protein:,} cells x {n_proteins} proteins")
print(f"  Match rate: {100*n_with_protein/n_rna_cells:.1f}%")
print(f"")
print(f"  Protein target: DSB difference (cell - NTC)")
print(f"  Protein effect range: [{matched_effects.min():.2f}, {matched_effects.max():.2f}]")
print(f"  Protein effect std: {matched_effects.std():.3f}")
print(f"  RNA log2FC std: {rna_matched.std():.3f}")
print(f"  Scale ratio (RNA/Protein): {rna_matched.std()/matched_effects.std():.2f}")
print(f"")
print(f"  Output: {OUTPUT_PATH}")
print(f"  Size: {file_size:.1f} MB")
print(f"")
print(f"  → Next: CDT_Morris_Trimodal_Training.ipynb")
print(f"")
print(f"  NOTE: Scale ratioを見てλ (protein_loss_weight) を調整:")
print(f"    λ ≈ (RNA_std / Protein_std)^2 が目安")
print(f"    現在の設定: λ = 1.0 (訓練ノートブックで調整可)")
print("=" * 60)